# Gold 06 — Saved Model Validation / Test-Only Review

This notebook validates saved Gold model artifacts without retraining. It should be run after Gold 01 through Gold 05 have produced model files, threshold summaries, comparison artifacts, and the Gold test split. The purpose is to prove that the trained model artifacts can be loaded from disk and scored against held-out data as a separate validation step.


## 1. Imports

This cell imports the libraries needed for saved-model validation. The notebook uses `joblib` for model loading, pandas/numpy for scoring inputs, and scikit-learn metrics for the validation table. It also imports the shared notebook context loader so Gold 06 follows the same setup pattern as the other Gold notebooks.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Dict, List

import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from utils.core.notebook_context import load_notebook_context
from utils.core.logging_setup import log_layer_paths
from utils.core.file_io import save_json


## 2. Shared notebook context

This cell loads the shared Gold validation context. Gold 06 uses `mode="test"` because it should load saved model files and score the test split instead of fitting new estimators. The aliases keep the same notebook variable style used by Gold 01 through Gold 05.


In [ ]:
# Shared notebook context
CONTEXT_STAGE = "gold_model_validation"
CONTEXT_DATASET = "pump"
CONTEXT_LAYER = "gold"
CONFIG_RUN_MODE = "test"
CONFIG_PROFILE = "default"
CONTEXT_LOG_FILE = "gold_model_validation.log"

CTX = load_notebook_context(
    stage=CONTEXT_STAGE,
    dataset=CONTEXT_DATASET,
    mode=CONFIG_RUN_MODE,
    profile=CONFIG_PROFILE,
    logger_child_name="capstone.gold.model_validation",
    log_filename=CONTEXT_LOG_FILE,
)

# Shared aliases used throughout the notebook
paths = CTX.paths
CONFIG = CTX.config
CONFIG_MAP = CTX.config
STAGE_CFG = CTX.stage_config
GOLD_CFG = CTX.stage_config
GOLD_VALIDATION_CFG = CTX.stage_config
RESOLVED_PATHS = CTX.resolved_paths
FILENAMES = CTX.filenames
VERSIONS_CFG = CTX.versions
RUNTIME_CFG = CTX.runtime
DATASET_CFG = CTX.dataset_config
WANDB_CFG = CTX.wandb
EXECUTION_CFG = CTX.execution
PIPELINE = CTX.pipeline
logger = CTX.logger
ledger = CTX.ledger
LOG_PATH = CTX.log_path
CONTEXT_RECIPE_ID = CTX.recipe_id

logger.info(
    "Notebook context loaded",
    extra={
        "stage": CONTEXT_STAGE,
        "recipe_id": CONTEXT_RECIPE_ID,
        "dataset": CONTEXT_DATASET,
        "mode": CONFIG_RUN_MODE,
        "profile": CONFIG_PROFILE,
    },
)

ledger.add(
    kind="step",
    step="context_loaded",
    message="Loaded shared Gold 06 saved-model validation context.",
    data={
        "stage": CONTEXT_STAGE,
        "recipe_id": CONTEXT_RECIPE_ID,
        "dataset": CONTEXT_DATASET,
        "mode": CONFIG_RUN_MODE,
        "profile": CONFIG_PROFILE,
        "log_path": str(LOG_PATH),
    },
    logger=logger,
)


## 3. Context sanity check

This cell verifies that the shared context variables needed by the notebook are present. This is a small defensive check before loading models or reading artifacts. If the YAML file is missing or the context loader is not installed correctly, this cell should fail before any scoring work begins.


In [ ]:
required_context_vars = [
    "CTX",
    "paths",
    "CONFIG",
    "CONFIG_MAP",
    "STAGE_CFG",
    "GOLD_CFG",
    "GOLD_VALIDATION_CFG",
    "RESOLVED_PATHS",
    "FILENAMES",
    "VERSIONS_CFG",
    "RUNTIME_CFG",
    "DATASET_CFG",
    "WANDB_CFG",
    "EXECUTION_CFG",
    "PIPELINE",
    "logger",
    "ledger",
    "LOG_PATH",
]

missing_context_vars = [name for name in required_context_vars if name not in globals()]
if missing_context_vars:
    raise NameError(f"Missing required context variables: {missing_context_vars}")

log_layer_paths(paths, current_layer=CONTEXT_LAYER, logger=logger)

logger.info(
    "Context sanity check passed",
    extra={
        "stage": CTX.stage,
        "recipe_id": CTX.recipe_id,
        "dataset": CTX.dataset,
        "mode": CTX.mode,
        "profile": CTX.profile,
        "log_path": str(CTX.log_path),
    },
)

ledger.add(
    kind="check",
    step="context_sanity_check",
    message="Verified shared notebook context variables are available.",
    data={
        "stage": CTX.stage,
        "recipe_id": CTX.recipe_id,
        "dataset": CTX.dataset,
        "mode": CTX.mode,
        "profile": CTX.profile,
        "log_path": str(CTX.log_path),
    },
    logger=logger,
)

pd.DataFrame([{"name": name, "status": "present"} for name in required_context_vars])


## 4. Resolve validation paths from config

This cell resolves input and output paths for Gold 06. The input side points to saved models, threshold JSON files, and test data candidates. The output side creates a dedicated `artifacts/gold/pump/model_validation/` folder so validation outputs do not mix with training artifacts.


In [ ]:
PROJECT_ROOT = paths.root
DATA_ROOT = paths.data
ARTIFACTS_ROOT = paths.artifacts
MODELS_ROOT = paths.models

VALIDATION_ROOT = ARTIFACTS_ROOT / "gold" / CONTEXT_DATASET / "model_validation"
VALIDATION_RESULTS_DIR = VALIDATION_ROOT / "results"
VALIDATION_SCORES_DIR = VALIDATION_ROOT / "scores"
VALIDATION_SUMMARY_DIR = VALIDATION_ROOT / "summaries"
VALIDATION_METADATA_DIR = VALIDATION_ROOT / "metadata"
VALIDATION_CONFIG_DIR = VALIDATION_ROOT / "config"
VALIDATION_LINEAGE_DIR = VALIDATION_ROOT / "lineage"

for directory in [
    VALIDATION_RESULTS_DIR,
    VALIDATION_SCORES_DIR,
    VALIDATION_SUMMARY_DIR,
    VALIDATION_METADATA_DIR,
    VALIDATION_CONFIG_DIR,
    VALIDATION_LINEAGE_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

input_cfg = dict(GOLD_VALIDATION_CFG.get("inputs", {}) or {})
model_cfg = dict(GOLD_VALIDATION_CFG.get("models", {}) or {})
output_cfg = dict(GOLD_VALIDATION_CFG.get("outputs", {}) or {})
validation_cfg = dict(GOLD_VALIDATION_CFG.get("validation", {}) or {})

def project_path(path_value: str | Path) -> Path:
    candidate = Path(path_value)
    if candidate.is_absolute():
        return candidate
    return PROJECT_ROOT / candidate

TEST_DATA_CANDIDATES = [
    project_path(value)
    for value in input_cfg.get(
        "test_data_candidates",
        [
            "data/gold/pump__gold__test.parquet",
            "data/gold/pump__gold_test.parquet",
            "data/gold/pump__gold__model_test.parquet",
        ],
    )
]

MODEL_SPECS = {
    model_name: {
        **spec,
        "model_path": project_path(spec["model_path"]),
        "threshold_path": project_path(spec["threshold_path"]),
    }
    for model_name, spec in model_cfg.items()
}

validation_paths = {
    "validation_root": str(VALIDATION_ROOT),
    "test_data_candidates": [str(path) for path in TEST_DATA_CANDIDATES],
    "model_count": len(MODEL_SPECS),
}

logger.info("Resolved Gold 06 validation paths", extra=validation_paths)
ledger.add(
    kind="step",
    step="validation_paths_resolved",
    message="Resolved saved-model validation input and output paths.",
    data=validation_paths,
    logger=logger,
)

pd.DataFrame(
    [
        {
            "model_name": name,
            "model_path": str(spec["model_path"]),
            "model_exists": spec["model_path"].exists(),
            "threshold_path": str(spec["threshold_path"]),
            "threshold_exists": spec["threshold_path"].exists(),
            "variant_family": spec.get("variant_family"),
            "stage_gate": spec.get("stage_gate"),
        }
        for name, spec in MODEL_SPECS.items()
    ]
)


## 5. Load held-out test split

This cell loads the saved Gold test split. Gold 06 should not recreate the train/test split because the point is to validate saved artifacts. If this cell cannot find a test split, stop here and update Gold 01 so it exports a durable test dataframe.


In [ ]:
test_path = next((path for path in TEST_DATA_CANDIDATES if path.exists()), None)

if test_path is None:
    raise FileNotFoundError(
        "Could not find the Gold test split. Checked: "
        + "; ".join(str(path) for path in TEST_DATA_CANDIDATES)
    )

test_dataframe = pd.read_parquet(test_path)

logger.info(
    "Loaded Gold test split",
    extra={"path": str(test_path), "rows": len(test_dataframe), "columns": len(test_dataframe.columns)},
)
ledger.add(
    kind="step",
    step="test_split_loaded",
    message="Loaded held-out Gold test split for saved-model validation.",
    data={"path": str(test_path), "shape": list(test_dataframe.shape)},
    logger=logger,
)

display(test_dataframe.head(3))


## 6. Prepare labels and feature matrix

This cell identifies the binary target column and numeric model feature columns. It excludes metadata, status, label, score, alert, and prediction columns so the validation matrix is aligned with model input features rather than downstream outputs.


In [ ]:
LABEL_CANDIDATES = [
    "target_flag",
    "is_anomaly",
    "is_broken",
    "broken_flag",
    "target",
    "label",
]

label_column = next((column for column in LABEL_CANDIDATES if column in test_dataframe.columns), None)
if label_column is None:
    raise KeyError(f"Could not find binary target label column. Checked: {LABEL_CANDIDATES}")

EXCLUDE_PATTERNS = (
    "target",
    "label",
    "status",
    "timestamp",
    "datetime",
    "date",
    "split",
    "run_id",
    "dataset_id",
    "prediction",
    "pred",
    "alert",
    "score",
    "anomaly",
    "broken",
    "recovering",
)

feature_columns = [
    column
    for column in test_dataframe.select_dtypes(include=[np.number]).columns
    if column != label_column
    and not any(pattern in str(column).lower() for pattern in EXCLUDE_PATTERNS)
]

if not feature_columns:
    raise ValueError("No numeric model feature columns were found after exclusions.")

y_test = test_dataframe[label_column].astype(int).to_numpy()
X_test = (
    test_dataframe[feature_columns]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0.0)
)

logger.info(
    "Prepared validation feature matrix",
    extra={"label_column": label_column, "feature_count": len(feature_columns), "rows": len(X_test)},
)
ledger.add(
    kind="step",
    step="validation_features_prepared",
    message="Prepared validation features and labels from the held-out Gold test split.",
    data={"label_column": label_column, "feature_count": len(feature_columns), "rows": len(X_test)},
    logger=logger,
)

pd.DataFrame({"feature_column": feature_columns}).head(25)


## 7. Load saved models and thresholds

This cell loads model artifacts and threshold JSON files. It intentionally does not call `.fit()`. Missing model files are treated as a failure because this notebook validates prior training outputs.


In [ ]:
models: Dict[str, Any] = {}
missing_models: list[str] = []
thresholds: Dict[str, Dict[str, Any]] = {}
missing_thresholds: list[str] = []

for model_name, spec in MODEL_SPECS.items():
    model_path = spec["model_path"]
    threshold_path = spec["threshold_path"]

    if model_path.exists():
        models[model_name] = joblib.load(model_path)
    else:
        missing_models.append(str(model_path))

    if threshold_path.exists():
        with threshold_path.open("r", encoding="utf-8") as file:
            thresholds[model_name] = json.load(file)
    else:
        missing_thresholds.append(str(threshold_path))

if missing_models:
    raise FileNotFoundError("Missing saved model files: " + "; ".join(missing_models))

if missing_thresholds and bool(validation_cfg.get("require_threshold_files", False)):
    raise FileNotFoundError("Missing threshold files: " + "; ".join(missing_thresholds))

logger.info(
    "Loaded saved models and thresholds",
    extra={"models": sorted(models), "threshold_sets": sorted(thresholds), "missing_thresholds": missing_thresholds},
)
ledger.add(
    kind="step",
    step="saved_models_loaded",
    message="Loaded saved Gold model artifacts for test-only validation.",
    data={"model_count": len(models), "threshold_count": len(thresholds), "missing_threshold_count": len(missing_thresholds)},
    logger=logger,
)

pd.DataFrame(
    [
        {
            "model_name": name,
            "loaded": name in models,
            "threshold_loaded": name in thresholds,
            "variant_family": MODEL_SPECS[name].get("variant_family"),
            "stage_gate": MODEL_SPECS[name].get("stage_gate"),
        }
        for name in MODEL_SPECS
    ]
)


## 8. Define scoring and metric helpers

These helpers convert Isolation Forest outputs into positive anomaly-strength scores, resolve thresholds, and calculate binary metrics. They are kept in one cell so the validation math is easy to inspect and compare with the training notebooks.


In [ ]:
def anomaly_score(model: Any, X: pd.DataFrame) -> np.ndarray:
    """Return a positive anomaly-strength score for an Isolation Forest-like model.

    scikit-learn Isolation Forest scores lower for more abnormal observations.
    This helper flips the sign so larger values mean stronger anomaly evidence.
    That makes validation thresholds easier to read and keeps scoring outputs
    consistent across saved models.
    """
    if hasattr(model, "decision_function"):
        return -model.decision_function(X)
    if hasattr(model, "score_samples"):
        return -model.score_samples(X)
    raise TypeError(f"Model {type(model).__name__} does not expose decision_function or score_samples")


def threshold_from_summary(threshold_payload: Dict[str, Any], fallback_quantile: float, scores: np.ndarray) -> float:
    """Resolve a numeric alert threshold from a threshold payload or fallback quantile.

    The Gold notebooks store threshold information under a few different names.
    This function checks common keys first. If none are available, it falls back
    to a quantile of the validation scores so individual model scoring can still
    run while the exact cascade-gating threshold contract is standardized.
    """
    possible_keys = [
        "threshold",
        "score_threshold",
        "baseline_threshold",
        "stage1_threshold",
        "stage2_threshold",
        "final_threshold",
    ]

    for key in possible_keys:
        value = threshold_payload.get(key)
        if isinstance(value, (int, float)):
            return float(value)

    return float(np.quantile(scores, fallback_quantile))


def calculate_binary_metrics(
    *,
    model_name: str,
    model_spec: Dict[str, Any],
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_score: np.ndarray,
    threshold: float,
) -> Dict[str, Any]:
    """Calculate validation metrics for one saved model against the test split."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    metric_row = {
        "model_id": model_name,
        "variant_family": model_spec.get("variant_family"),
        "stage_gate": model_spec.get("stage_gate"),
        "test_rows": int(len(y_true)),
        "alerts": int(y_pred.sum()),
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "threshold": float(threshold),
    }

    try:
        metric_row["roc_auc"] = float(roc_auc_score(y_true, y_score))
    except ValueError:
        metric_row["roc_auc"] = None

    try:
        metric_row["average_precision"] = float(average_precision_score(y_true, y_score))
    except ValueError:
        metric_row["average_precision"] = None

    return metric_row


## 9. Score saved models against test split

This cell scores each saved model independently against the held-out test split. This is not yet an exact full cascade replay unless the stage-gating artifacts are standardized, but it is a valid saved-model loading and test-only scoring check.


In [ ]:
validation_rows: list[dict[str, Any]] = []
score_frames: list[pd.DataFrame] = []

for model_name, model in models.items():
    spec = MODEL_SPECS[model_name]
    scores = anomaly_score(model, X_test)
    threshold = threshold_from_summary(
        thresholds.get(model_name, {}),
        fallback_quantile=0.95,
        scores=scores,
    )
    predictions = (scores >= threshold).astype(int)

    validation_rows.append(
        calculate_binary_metrics(
            model_name=model_name,
            model_spec=spec,
            y_true=y_test,
            y_pred=predictions,
            y_score=scores,
            threshold=threshold,
        )
    )

    score_frames.append(
        pd.DataFrame(
            {
                "row_index": np.arange(len(test_dataframe)),
                "model_id": model_name,
                "variant_family": spec.get("variant_family"),
                "stage_gate": spec.get("stage_gate"),
                "score": scores,
                "prediction": predictions,
                "label": y_test,
            }
        )
    )

validation_dataframe = pd.DataFrame(validation_rows).sort_values(
    ["f1", "precision", "recall"],
    ascending=False,
).reset_index(drop=True)

validation_scores_dataframe = pd.concat(score_frames, ignore_index=True)

logger.info(
    "Scored saved models against test split",
    extra={"model_count": len(models), "score_rows": len(validation_scores_dataframe)},
)
ledger.add(
    kind="step",
    step="saved_models_scored",
    message="Scored saved models against the held-out Gold test split.",
    data={"model_count": len(models), "score_rows": len(validation_scores_dataframe)},
    logger=logger,
)

validation_dataframe


## 10. Compare validation output to Gold 04 when available

This cell compares saved-model validation rows to the Gold 04 comparison artifact if it exists. The comparison is diagnostic only. Gold 04 remains the report source of truth until Gold 06 exactly reconstructs cascade gates and operating modes.


In [ ]:
GOLD04_COMPARISON_PATH = ARTIFACTS_ROOT / "gold" / CONTEXT_DATASET / "comparison" / "results" / "pump__gold__model_comparison.csv"

if GOLD04_COMPARISON_PATH.exists():
    gold04_comparison_dataframe = pd.read_csv(GOLD04_COMPARISON_PATH)
    validation_vs_gold04_dataframe = validation_dataframe.merge(
        gold04_comparison_dataframe,
        how="left",
        left_on="model_id",
        right_on="model_id",
        suffixes=("_validation", "_gold04"),
    )
else:
    gold04_comparison_dataframe = pd.DataFrame()
    validation_vs_gold04_dataframe = validation_dataframe.copy()
    validation_vs_gold04_dataframe["gold04_available"] = False

logger.info(
    "Compared saved-model validation output to Gold 04 artifact",
    extra={"gold04_available": GOLD04_COMPARISON_PATH.exists(), "rows": len(validation_vs_gold04_dataframe)},
)
ledger.add(
    kind="check",
    step="validation_vs_gold04_check",
    message="Compared saved-model validation metrics with Gold 04 artifact when available.",
    data={"gold04_available": GOLD04_COMPARISON_PATH.exists(), "rows": len(validation_vs_gold04_dataframe)},
    logger=logger,
)

validation_vs_gold04_dataframe.head(20)


## 11. Save validation outputs

This cell writes the validation metric table, row-level score table, Gold 04 comparison alignment table, and JSON summary. These artifacts give the project a formal test-only validation stage separate from the training notebooks.


In [ ]:
metrics_file = output_cfg.get("metrics_file", "pump__gold__saved_model_validation_metrics.csv")
scores_file = output_cfg.get("scores_file", "pump__gold__saved_model_validation_scores.csv")
summary_file = output_cfg.get("summary_file", "pump__gold__saved_model_validation_summary.json")
alignment_file = output_cfg.get("comparison_alignment_file", "pump__gold__saved_model_validation_vs_gold04.csv")

validation_table_path = VALIDATION_RESULTS_DIR / metrics_file
validation_scores_path = VALIDATION_SCORES_DIR / scores_file
validation_alignment_path = VALIDATION_RESULTS_DIR / alignment_file
validation_summary_path = VALIDATION_SUMMARY_DIR / summary_file

validation_dataframe.to_csv(validation_table_path, index=False)
validation_scores_dataframe.to_csv(validation_scores_path, index=False)
validation_vs_gold04_dataframe.to_csv(validation_alignment_path, index=False)

summary_payload = {
    "stage": CONTEXT_STAGE,
    "recipe_id": CONTEXT_RECIPE_ID,
    "mode": CONFIG_RUN_MODE,
    "profile": CONFIG_PROFILE,
    "test_data_path": str(test_path),
    "test_rows": int(len(test_dataframe)),
    "label_column": label_column,
    "feature_count": int(len(feature_columns)),
    "models_validated": sorted(models),
    "metrics_output": str(validation_table_path),
    "scores_output": str(validation_scores_path),
    "gold04_alignment_output": str(validation_alignment_path),
    "gold04_comparison_available": bool(GOLD04_COMPARISON_PATH.exists()),
    "note": (
        "Gold 06 validates saved model loading and test-only scoring. "
        "Gold 04 remains the final report comparison source until Gold 06 exactly reconstructs full cascade gating."
    ),
}

validation_summary_path.write_text(json.dumps(summary_payload, indent=2), encoding="utf-8")

logger.info(
    "Saved model validation outputs",
    extra={
        "metrics": str(validation_table_path),
        "scores": str(validation_scores_path),
        "summary": str(validation_summary_path),
    },
)
ledger.add(
    kind="result",
    step="saved_model_validation_outputs",
    message="Saved Gold 06 validation metrics, scores, alignment table, and summary JSON.",
    data=summary_payload,
    logger=logger,
)

validation_table_path, validation_scores_path, validation_alignment_path, validation_summary_path


## 12. Validation interpretation

This final cell summarizes the test-only validation run. The key interpretation is whether saved model files can be loaded and scored without retraining. Do not replace Gold 04 report numbers with this table until the full cascade-gating replay is implemented.


In [ ]:
best_f1 = validation_dataframe.iloc[0].to_dict() if not validation_dataframe.empty else {}
lowest_alert = validation_dataframe.sort_values("alerts").iloc[0].to_dict() if not validation_dataframe.empty else {}

interpretation = {
    "best_f1_model": best_f1.get("model_id"),
    "best_f1": best_f1.get("f1"),
    "lowest_alert_model": lowest_alert.get("model_id"),
    "lowest_alert_count": lowest_alert.get("alerts"),
    "report_source_of_truth": "Gold 04 remains the main model comparison source until exact cascade replay is implemented here.",
}

logger.info("Completed saved-model validation interpretation", extra=interpretation)
ledger.add(
    kind="result",
    step="saved_model_validation_complete",
    message="Completed Gold 06 saved-model validation notebook.",
    data=interpretation,
    logger=logger,
)

interpretation
